# Arabic Diacritization — Preprocessing for Fine-Tuning Qwen3.5-9B

**Training data:** `Misraj/Sadeed_Tashkeela` (`train` / `test` splits, columns: `filename`, `input`, `output`)
**Evaluation benchmark:** `Misraj/SadeedDiac-25`

This notebook covers only:
1. Setup & Hugging Face authentication
2. Loading the dataset
3. Building instruction/QA-formatted (`messages`) training examples — Sadeed §5
4. Light preprocessing of `Misraj/SadeedDiac-25` (the benchmark)

Everything stays in memory as 🤗 `Dataset` objects — nothing is written to disk.


## 1. Setup & Hugging Face Authentication

Set your token in the `HF_TOKEN` environment variable (recommended) or paste it directly into
`HF_TOKEN = "..."` below. Never commit a real token to source control.


In [ ]:
# pip install -q datasets huggingface_hub

import os
from datasets import load_dataset
from huggingface_hub import login

# --- Hugging Face auth -------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN")  # or: HF_TOKEN = "hf_xxx..."

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN found. Set the HF_TOKEN environment variable, or run "
          "`huggingface-cli login` in a terminal, before running the next cell.")


In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Misraj/Sadeed_Tashkeela")
print(ds)

train_raw = ds["train"]
test_raw = ds["test"]

print(train_raw[0])


In [ ]:
# Confirmed columns: [filename, output, input]
INPUT_COLUMN = "input"     # undiacritized source text
OUTPUT_COLUMN = "output"   # fully diacritized target text


## 2. Build Instruction/QA-formatted (chat) examples — Sadeed §5

Sadeed reformulates diacritization as a QA task for its decoder-only base model: a fixed
**system prompt**, the raw sentence as the **user** turn, and the diacritized sentence as the
**assistant** turn — trained with the loss masked on everything except the assistant turn
(Appendix A: *"system prompt and embedding tokens masked"*).

Since the dataset already provides clean `input`/`output` pairs, we don't need to strip diacritics
ourselves — we just wrap them in Qwen's chat format.


In [ ]:
SYSTEM_PROMPT = (
    "أنت نظام متخصص في التشكيل الآلي للنصوص العربية. "
    "مهمتك إضافة الحركات (التشكيل) الصحيحة إلى النص العربي المُدخل دون تغيير الكلمات أو ترتيبها، "
    "مع مراعاة السياق النحوي والصرفي الكامل للجملة."
)
# Written independently for this notebook; inspired by (not a reproduction of) the QA-style
# system prompt described in Sadeed §5 / Figure 3.


def build_chat_example(ex):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex[INPUT_COLUMN]},
            {"role": "assistant", "content": ex[OUTPUT_COLUMN]},
        ]
    }


train_chat = train_raw.map(build_chat_example, remove_columns=train_raw.column_names)
test_chat = test_raw.map(build_chat_example, remove_columns=test_raw.column_names)

print(train_chat[0])


In [ ]:
# At training time, render `messages` through Qwen's tokenizer chat template:
#
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-9B")
#
# def to_text(ex):
#     return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False,
#                                                     add_generation_prompt=False)}
#
# train_text = train_chat.map(to_text)
#
# train_chat / test_chat are ready to hand directly to e.g. TRL's SFTTrainer —
# no JSON export needed, they stay as in-memory `datasets.Dataset` objects.


## 3. `Misraj/SadeedDiac-25` — light preprocessing only

> **Note:** `SadeedDiac-25` is the fixed, expert-reviewed evaluation benchmark (Sadeed §4). It
> receives **only light preprocessing here** — building the same `messages` chat structure so it
> can be fed straight into your evaluation/generation loop. It is **not** cleaned with any custom
> normalization rules, **not** chunked, **not** filtered, and **not** deduplicated against
> anything. Every paragraph must remain exactly as Misraj curated it, or your reported DER/WER
> numbers stop being comparable to the numbers reported in the Sadeed paper.


In [ ]:
import re

FATHA, DAMMA, KASRA   = "\u064E", "\u064F", "\u0650"
SHADDA, SUKUN         = "\u0651", "\u0652"
FATHATAN, DAMMATAN, KASRATAN = "\u064B", "\u064C", "\u064D"
ALL_DIACRITICS = {FATHA, DAMMA, KASRA, SHADDA, SUKUN, FATHATAN, DAMMATAN, KASRATAN}
DIACRITIC_RE = re.compile("[" + "".join(ALL_DIACRITICS) + "]")


def strip_diacritics(text: str) -> str:
    return DIACRITIC_RE.sub("", text)


benchmark_ds = load_dataset("Misraj/SadeedDiac-25")
print(benchmark_ds)

# Inspect one example to confirm the benchmark's split/column name(s) before proceeding.
print(benchmark_ds[list(benchmark_ds.keys())[0]][0])


In [ ]:
# Adjust after inspecting the cell above.
BENCHMARK_SPLIT = "test"          # or "train" / whatever split SadeedDiac-25 exposes
BENCHMARK_TEXT_COLUMN = "output"  # diacritized ground truth column name

benchmark_raw = benchmark_ds[BENCHMARK_SPLIT]


def build_benchmark_chat_example(ex):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": strip_diacritics(ex[BENCHMARK_TEXT_COLUMN])},
        ],
        "reference": ex[BENCHMARK_TEXT_COLUMN],  # ground truth, kept separately for DER/WER scoring
    }


benchmark_chat = benchmark_raw.map(build_benchmark_chat_example)
print(benchmark_chat[0])

# benchmark_chat stays in memory; feed `messages` to the model for generation and compare its
# output against `reference` using DER/WER (with/without case-ending, with/without no-diacritic
# characters — see the 4-way breakdown in Sadeed Table 6-8 / PTCAD Table 7-8).
